In [1]:
import numpy as np
import torch
import scanpy as sc
import anndata as ad
import os
import pandas as pd
from utils.preprocess import *

In [2]:
import os
import sys
    
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
import torch
import torchsde
from torchdyn.core import NeuralODE
from tqdm import tqdm

from torchcfm.conditional_flow_matching import *
from torchcfm.models import MLP
from torchcfm.utils import plot_trajectories, torch_wrapper
from simulate.simulate import *
from omegaconf import OmegaConf
from utils.hydra import *
from datasets.process import *
from scripts.run_model import *

In [3]:
import matplotlib.pyplot as plt
%matplotlib inline

In [4]:
%reload_ext autoreload
%autoreload 2

In [5]:
############################################################

In [9]:
import os

current_directory = os.getcwd()
print(current_directory)

/home/azweig/projects/finfm


In [15]:
import os
path = "data"
suffix = "pairwise_hvg.h5ad"
filename = os.path.join(path, suffix)
adata = load_data(filename)

In [16]:
adata

View of AnnData object with n_obs × n_vars = 2686378 × 4569
    obs: 'cell', 'Size_Factor', 'n.umi', 'perc_mitochondrial_umis', 'timepoint', 'hash_umis', 'top_to_second_best_ratio', 'expt', 'cell_type_sub', 'cell_type_broad', 'tissue', 'germ_layer', 'log.n.umi', 'num_genes_expressed', 'umap3d_1', 'umap3d_2', 'umap3d_3', 'major_group', 'gene_target', 'mean_nn_time', 'subumap3d_1', 'subumap3d_2', 'subumap3d_3', 'embryo', 'temp', 'n_genes'
    var: 'gene_short_name', 'id', 'chromosome', 'bp1', 'bp2', 'gene_strand', 'num_cells_expressed', 'n_cells', 'highly_variable'
    uns: 'log1p'

In [14]:
### SETTINGS ###

from collections import namedtuple
Holdout = namedtuple('Holdout', ['t', 'gene'])
holdout = Holdout(24, 'ctrl-inj')

config = load_config()
OmegaConf.set_struct(config, False)
config.pc_dim = 100

adata = process_data(pc_dim=config.pc_dim, data="zebrafish")
config.num_classes = adata.obs['cell_type'].nunique()

config.metric = "mfm"
config.finsler.use = True
config.finsler.lamb = 3.0

project = "zebrafish"

/home/azweig/miniconda3/envs/spatialenv/lib/python3.10/site-packages/scanpy/preprocessing/_pca/__init__.py:379: ImplicitModificationWarning: Setting element `.obsm['X_pca']` of view, initializing view as actual.
  adata.obsm[key_obsm] = X_pca


KeyError: 'cell_type'

In [ ]:
#TODO: fix + make ctrl make sense
test_bool = (adata.obs['timepoint'] == holdout.t) & (adata.obs['gene_target'] == holdout.gene)
adata_train = adata[~test_bool]
adata_test = adata[test_bool]

AnnData object with n_obs × n_vars = 4998 × 2
    obs: 'timepoint', 'gene_target', 'cell_type', 'cell_type_one_hot'
    uns: 'std', 'tree'
    obsm: 'X_pca'
    varm: 'PCs'

In [9]:
dataset = extract_paired_dataset(adata)

In [10]:
############################################################

In [12]:
classifier_model, metric_model, embed_model, flow_model = run_full_model(config, project, adata, dataset)

Running phase classifier:.......


wandb: Currently logged in as: az831 (az831-new-york-genome-center) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/azweig/miniconda3/envs/spatialenv/lib/python3.10/site-packages/pytorch_lightning/loggers/wandb.py:397: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.
wandb: logging graph, to disable use `wandb.watch(log_graph=False)`
You are using a CUDA device ('NVIDIA GeForce RTX 4080 SUPER') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name           | Type           | Params | Mode 
----------------------------------------------------

### DEBUG: classifier is constrained to linear ###


/home/azweig/miniconda3/envs/spatialenv/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=23` in the `DataLoader` to improve performance.
`Trainer.fit` stopped: `max_epochs=500` reached.


...

/home/azweig/miniconda3/envs/spatialenv/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=23` in the `DataLoader` to improve performance.
`Trainer.fit` stopped: `max_epochs=3000` reached.


epoch,▁▁▁▁▁▁▂▂▂▂▂▂▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇██
train_loss_embed,▅▇▆█▄█▅▆▄▃▄▆▇▇▆▆▇▆▅▄▄▄▃▃▃▂▂▂▃▂▃▂▂▁▆▂▁▂▁▁
train_loss_geo,▆▆▆▇▇▇▆▇▇▆▆▆▆▆██▅▅▆▅▅▅▄▄▃▃▃▃▃▂▂▁▂▄▃▁▃▂▄▁
trainer/global_step,▁▁▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▃▄▄▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇███
epoch,2999
train_loss_embed,0.89239
train_loss_geo,9.70335
trainer/global_step,2999


Running phase flow:.......


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/azweig/miniconda3/envs/spatialenv/lib/python3.10/site-packages/pytorch_lightning/loggers/wandb.py:397: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.
wandb: logging graph, to disable use `wandb.watch(log_graph=False)`
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name        | Type                     | Params | Mode 
-----------------------------------------------------------------
0 | flow_net    | SinNet                   | 217 K  | train
1 | embed_model | FinslerEmbedNetTrainBase | 646 K  | eval 
-----------------------------------------------------------------
217 K     Trainable params
646 K     Non-trainable params
863 K     Total params
3.452     Total estimated model params size (MB)
16        Modules in train mode
5

epoch,▁▁▁▂▂▂▃▃▄▄▄▄▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇█
train_loss,█▂▂▂▂▂▂▂▁▂▂▁▁▂▁▁▁▁▂▁▁▁▂▁▁▁▂▂▂▁▁▁▁▂▁▂▁▁▁▁
trainer/global_step,▁▁▁▁▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▅▅▅▅▆▆▆▆▆▇▇███
epoch,9999
train_loss,0.06599
trainer/global_step,9999


In [13]:
#fix the wandb.run.summary bug?
def remove_all_forward_hooks(model):
    for module in model.modules():
        module._forward_hooks.clear()

remove_all_forward_hooks(classifier_model)
remove_all_forward_hooks(metric_model)
remove_all_forward_hooks(embed_model)
remove_all_forward_hooks(flow_model)

In [ ]:
value = holdout.gene
t = holdout.t
num_traj = 10000
# num_traj = 100

print(predict(flow_model, adata_raw, value, conditions, num_traj, t))
print(predict_ctrl(flow_model, adata_raw, value, conditions, num_traj, t)) #DEBUG
print(predict_exact(flow_model, adata_raw, value, conditions, num_traj, t)) #DEBUG